# Regime-Aware GBWM — reproducible pipeline

Companion notebook to the CME 241 proposal. Runs the whole experiment:
**simulate market → baselines + G-Learner + Regime-Aware + Q-Learner → compare → learned policy → Q-learning convergence.**

> Educational simulation — not financial advice.


## 0 · Setup
Install once with `pip install -e ".[all,dev]"`. If running from a source checkout without installing, uncomment the `sys.path` line.


In [ ]:
import sys
# sys.path.insert(0, '../src')  # uncomment if 'gbwm' is not installed
import numpy as np
import matplotlib.pyplot as plt
from gbwm.config import default_config
from gbwm.simulation.regimes import MarketModel
from gbwm.policies import (BuyAndHold, SixtyForty, GlidePath,
                           GLearner, RegimeAwareGLearner, QLearner)
from gbwm.evaluation.harness import compare_policies, results_table
from gbwm.evaluation import plots
cfg = default_config(); cfg.simulation.n_episodes = 1000
print(f"goal: ${cfg.goal.initial_wealth:,.0f} -> ${cfg.goal.target_wealth:,.0f} "
      f"in {cfg.goal.horizon_years}y, +${cfg.goal.contribution:,.0f}/mo")


## 1 · The market (GBM + Markov-switching regimes)


In [ ]:
mm = MarketModel.from_config(cfg.market)
paths = mm.simulate(1, cfg.total_steps, np.random.default_rng(1))
price = 100 * np.cumprod(1 + paths.risky_returns[0, :, 0])
plt.figure(figsize=(8, 3)); plt.plot(price)
plt.title('One simulated market path'); plt.xlabel('month'); plt.ylabel('index'); plt.show()


## 2 · Strategies — baselines + the RL agents
Five strategies from the proposal's baseline slide, all behind one `Policy` interface.


In [ ]:
policies = {
    'Buy & Hold': BuyAndHold.from_config(cfg),
    '60/40': SixtyForty.from_config(cfg),
    'Glide Path': GlidePath.from_config(cfg),
    'G-Learner': GLearner.from_config(cfg),
    'Regime-Aware G-Learner': RegimeAwareGLearner.from_config(cfg),
}
list(policies)


## 3 · Evaluation — the proposal's metrics
Headline metric is **P(goal)**, alongside shortfall, terminal wealth, drawdown, turnover.


In [ ]:
results = compare_policies(policies, cfg)
results_table(results)


## 4 · The learned policy (regime-aware)
How much to hold in stocks across wealth × time — switch the `regime` argument (`bull`/`stable`/`high_vol`/`bear`) to see the agent de-risk in bad weather.


In [ ]:
ra = policies['Regime-Aware G-Learner']
plots.plot_policy_heatmap(ra, cfg.goal.target_wealth, cfg.steps_per_year, regime='bear')
plt.show()


## 5 · Q-learning learns by trial and error (TD)
A from-scratch agent's success rate climbing toward the exact G-learning solution.


In [ ]:
ql = QLearner.from_config(cfg)
curve = np.array(ql.learning_curve)
plt.figure(figsize=(8, 3))
plt.plot(curve[:, 0], curve[:, 1], '-o', label='Q-learner')
plt.axhline(results['G-Learner'].p_goal, ls='--', color='green', label='exact G-learning')
plt.title('Q-learning converging toward the exact solution')
plt.xlabel('training episodes'); plt.ylabel('P(goal)'); plt.legend(); plt.show()


## 6 · Deep RL (optional)
With the `rl` extra installed you can train PPO/SAC:
```python
from gbwm.policies.rl_agents import train_ppo_with_curve
ppo, curve = train_ppo_with_curve(cfg, total_timesteps=60_000)
```
